# 1. Embeddings

Calculates embeddings for the keywords database. To be ran on Kaggle.

Steps:
1. Add service_account.json secret with a service account with access to R2.
2. Enable accelerator 2x GPU T4
3. Save and commit so it runs on the background

It should take a little over 10 hours (2 minutes per partition for ~600 partitions / 2 GPUs)


In [ ]:
%pip install -q s3fs sentence-transformers[onnx-gpu] polars pyarrow

In [ ]:
import json
import multiprocessing as mp
import sys

import s3fs  # ty:ignore[unresolved-import]
from kaggle_secrets import UserSecretsClient  # ty:ignore[unresolved-import]

user_secrets = UserSecretsClient()
r2_creds = json.loads(user_secrets.get_secret("service_account.json"))
print("Authenticated with R2 access key:", r2_creds.get("access_key_id"))

R2_DATA_PATH = "keywordsdb/keywordsdb/2026-03-30"
R2_EMBEDDINGS_RAW = "keywordsdb/embeddings/raw"
FS = s3fs.S3FileSystem(
    key=r2_creds["access_key_id"], secret=r2_creds["secret_access_key"], endpoint_url=r2_creds["endpoint_url"]
)

In [3]:
def _get_input_files() -> list[str]:
    files = sorted(FS.glob(f"{R2_DATA_PATH}/*.parquet"))
    if not files:
        raise FileNotFoundError(f"No parquet files found in {R2_DATA_PATH}")
    return files


def _get_done_files(r2_dir: str) -> set[str]:
    try:
        return {f.split("/")[-1] for f in FS.glob(f"{r2_dir}/*.npz")}
    except FileNotFoundError:
        return set()


input_files = _get_input_files()
print(f"Input files: {len(input_files)}")
done = _get_done_files(R2_EMBEDDINGS_RAW)
remaining = [f for f in input_files if f.split("/")[-1].replace(".parquet", ".npz") not in done]
print(f"Embedding files: total={len(input_files)}, remaining={len(remaining)}")

In [ ]:
%%writefile /kaggle/working/_gpu_worker.py
import io
import os
import queue
import threading
import time

import numpy as np
import polars as pl
import s3fs


def run_on_gpu(gpu_id: int, file_paths: list, r2_creds: dict, r2_embeddings_raw: str) -> None:
    # Pin this process to one GPU before any CUDA/ONNX imports
    os.environ["CUDA_VISIBLE_DEVICES"] = str(gpu_id)

    from sentence_transformers import SentenceTransformer

    fs = s3fs.S3FileSystem(
        key=r2_creds["access_key_id"],
        secret=r2_creds["secret_access_key"],
        endpoint_url=r2_creds["endpoint_url"],
    )
    model = SentenceTransformer(
        "BAAI/bge-small-en-v1.5",
        backend="onnx",
        model_kwargs={"provider": "CUDAExecutionProvider"},
    )

    def _npz_save(r2_path, keywords, embeddings):
        buf = io.BytesIO()
        np.savez(buf, keywords=np.asarray(keywords), embeddings=embeddings)
        buf.seek(0)
        with fs.open(r2_path, "wb") as f:
            f.write(buf.read())

    def _prefetch_worker(paths, out_q):
        for fp in paths:
            with fs.open(fp, "rb") as f:
                df = pl.read_parquet(f, columns=["keyword"])
            out_q.put((fp, df))
        out_q.put(None)

    def _write_worker(write_q):
        while True:
            item = write_q.get()
            if item is None:
                break
            r2_path, keywords, embeddings = item
            _npz_save(r2_path, keywords=keywords, embeddings=embeddings)
            print(f"[GPU{gpu_id}] Saved: {r2_path.split('/')[-1]}, rows={len(keywords)}", flush=True)

    prefetch_q = queue.Queue(maxsize=3)
    write_q = queue.Queue(maxsize=3)
    prefetch_t = threading.Thread(target=_prefetch_worker, args=(file_paths, prefetch_q), daemon=True)
    write_t = threading.Thread(target=_write_worker, args=(write_q,), daemon=True)
    prefetch_t.start()
    write_t.start()

    files_encoded = 0
    t_start = time.perf_counter()
    while True:
        item = prefetch_q.get()
        if item is None:
            break

        file_path, df = item
        keywords = df["keyword"].to_list()
        name = file_path.split("/")[-1]
        t_file = time.perf_counter()

        emb_array = model.encode(
            keywords,
            batch_size=64,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=False,
        ).astype(np.float32)

        files_encoded += 1
        elapsed_file = time.perf_counter() - t_file
        elapsed_total = time.perf_counter() - t_start
        print(
            f"[GPU{gpu_id}] Encoded: {name}, progress={files_encoded}/{len(file_paths)}, "
            f"file={elapsed_file:.1f}s, total={elapsed_total:.0f}s",
            flush=True,
        )

        output_name = name.replace(".parquet", ".npz")
        write_q.put((f"{r2_embeddings_raw}/{output_name}", keywords, emb_array))

    write_q.put(None)
    prefetch_t.join()
    write_t.join()
    print(f"[GPU{gpu_id}] Done: {files_encoded} files in {time.perf_counter() - t_start:.0f}s", flush=True)


In [ ]:
sys.path.insert(0, "/kaggle/working")
from _gpu_worker import run_on_gpu  # ty:ignore[unresolved-import]

mp.set_start_method("spawn", force=True)

mid = len(remaining) // 2
splits = [remaining[:mid], remaining[mid:]]
print(f"Splitting {len(remaining)} files across 2 GPUs: {len(splits[0])} / {len(splits[1])}")

processes = [
    mp.Process(target=run_on_gpu, args=(gpu_id, subset, r2_creds, R2_EMBEDDINGS_RAW))
    for gpu_id, subset in enumerate(splits)
]

for p in processes:
    p.start()
for p in processes:
    p.join()

print("All GPUs done.")